In [1]:
import warnings
import os
warnings.simplefilter(action='ignore')
os.environ["PYTHONWARNINGS"] = "ignore"

In [2]:
#parameters

### USER EDIT start
esm_file='/g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/cm3-demo-datastore.json'
# esm_file= os.path.join(run_dir, 'cm3-demo-datastore/cm3-virtual-datastore.json')
plotfolder='/g/data/tm70/ek4684/access-om3-paper-1/notebooks/mkfigs_output4/'
dpi=300
### USER EDIT stop

import matplotlib as mpl
import os
%matplotlib inline
mpl.rcParams['figure.dpi']= dpi

os.makedirs(plotfolder, exist_ok=True)

 # a similar cell under this means it's being run in batch
print("ESM datastore path: ",esm_file)
print("Plot folder path: ",plotfolder)

ESM datastore path:  /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/cm3-demo-datastore.json
Plot folder path:  /g/data/tm70/ek4684/access-om3-paper-1/notebooks/mkfigs_output4/


In [3]:
import xarray as xr
import cf_xarray as cfxr
import intake
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from distributed import Client
import numpy as np
import dask.array as da
import iris

In [4]:
client = Client(threads_per_worker=1)
print(client.dashboard_link)

/proxy/34249/status


INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/atmos.1mon.bnds:2.depth:4.lat:144.lat_river:180.lat_v:145.lon:192.lon_river:360.lon_u:192.model_rho_level_number:85.model_theta_level_number:85.model_theta_level_number_0:50.model_theta_level_number_2:52.pressure:17.pseudo_level:6.pseudo_level_0:5.ps.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/atmos.1mon.bnds:2.depth:4.lat:144.lat_river:180.lat_v:145.lon:192.lon_river:360.lon_u:192.model_rho_level_number:85.model_theta_level_number:85.model_theta_level_number_0:50.model_theta_level_number_2:52.pressure:17.pseudo_level:6.pseudo_level_0:5.ps.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_o

## Load datastore and datasets

In [5]:
#datastore_path = "/g/data/ol01/access-om3-output/access-om3-025/MC_25km_jra_ryf-1.0-beta/experiment_datastore.json"
COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)

In [6]:
fn = "seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json"
fp = os.path.join(os.path.split(esm_file)[0], 'virtualised_outputs', fn)
cm3_ice_ds = xr.open_dataset(fp, engine="kerchunk")
cm3_ice_ds = cm3_ice_ds.rename(dict(ni='xh', nj='yh'))

INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json


In [7]:
fn = "atmos.1mon.bnds:2.depth:4.lat:144.lat_river:180.lat_v:145.lon:192.lon_river:360.lon_u:192.model_rho_level_number:85.model_theta_level_number:85.model_theta_level_number_0:50.model_theta_level_number_2:52.pressure:17.pseudo_level:6.pseudo_level_0:5.ps.json"
fp = os.path.join(os.path.split(esm_file)[0], 'virtualised_outputs', fn)
cm3_atm_ds = xr.open_dataset(fp, engine="kerchunk")

INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/atmos.1mon.bnds:2.depth:4.lat:144.lat_river:180.lat_v:145.lon:192.lon_river:360.lon_u:192.model_rho_level_number:85.model_theta_level_number:85.model_theta_level_number_0:50.model_theta_level_number_2:52.pressure:17.pseudo_level:6.pseudo_level_0:5.ps.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/atmos.1mon.bnds:2.depth:4.lat:144.lat_river:180.lat_v:145.lon:192.lon_river:360.lon_u:192.model_rho_level_number:85.model_theta_level_number:85.model_theta_level_number_0:50.model_theta_level_number_2:52.pressure:17.pseudo_level:6.pseudo_level_0:5.ps.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_o

## Load areas

In [8]:
wet = datastore.search(variable="wet").to_dask().compute()
areacello = datastore.search(variable="areacello").to_dask().compute()

areacello = (areacello.areacello * (wet.wet == 1.0))
ocn_area = areacello.sum().data

In [9]:
EARTH_RADIUS = 6371229.0
nlon, nlat = 192, 144
dx = 360 / nlon
dy = 180 / nlat

element_lat = (np.arange(nlat) + 0.5) * dy  - 90
element_lat = element_lat[:, None] * np.ones(nlon)[None, :]
element_lon = (np.arange(nlon) + 0.5) * dx

pi_over_180 = np.pi / 180
element_areas = dx * pi_over_180 * (
  np.sin((element_lat + 0.5 * dy) * pi_over_180) - np.sin((element_lat - 0.5 * dy) * pi_over_180)
)

areacella = element_areas * EARTH_RADIUS**2
areacella = xr.DataArray(areacella, coords=dict(lat=element_lat[:, 0], lon=element_lon), dims=('lat', 'lon'))

In [60]:
ancil_dir = '/scratch/tm70/kr4383/cylc-run/ancil-gen-30-07-2025/share/data/n96e_mom025_20250515/'
land_frac = iris.load_cube(os.path.join(ancil_dir, 'qrparm.landfrac')).data
vsat = iris.load_cube(os.path.join(ancil_dir, 'qrparm.soil'), 'soil_porosity').data
ocn_frac = 1 - land_frac

## Water flux error

In [29]:
%%time
xr_kw = dict(chunks={"yh": -1, "xh": -1}, decode_timedelta=True)

ocn_river = datastore.search(variable="friver", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).friver
ocn_evap = datastore.search(variable="evs", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).evs
ocn_snow = datastore.search(variable="prsn", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).prsn
ocn_rain = datastore.search(variable="prlq", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).prlq
ocn_total = datastore.search(variable="wfo", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).wfo

CPU times: user 4.11 s, sys: 726 ms, total: 4.83 s
Wall time: 12.3 s


In [55]:
ocn_river_avg = ocn_river.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / ocn_area
ocn_evap_avg = ocn_evap.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / ocn_area
ocn_snow_avg = ocn_snow.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / ocn_area
ocn_rain_avg = ocn_rain.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / ocn_area
ocn_total_avg = ocn_total.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / ocn_area
ocn_melt_avg = ocn_total_avg - (ocn_river_avg + ocn_evap_avg + ocn_snow_avg + ocn_rain_avg)

In [82]:
atm_river_avg = cm3_atm_ds.fld_s26i004.weighted(areacella).sum(dim=("lat", "lon")).compute() / ocn_area
atm_evap_avg = cm3_atm_ds.fld_s03i232.weighted(ocn_frac * areacella).sum(dim=("lat", "lon")).compute() / ocn_area
atm_rain_avg = cm3_atm_ds.fld_s05i214.weighted(ocn_frac * areacella).sum(dim=("lat", "lon")).compute() / ocn_area
atm_snow_avg = cm3_atm_ds.fld_s05i215.weighted(ocn_frac * areacella).sum(dim=("lat", "lon")).compute() / ocn_area
atm_sublim_avg = cm3_atm_ds.fld_s03i298.weighted(ocn_frac * areacella).sum(dim=("lat", "lon")).compute() / ocn_area

In [66]:
ice_snow_avg = cm3_ice_ds.snow_ai_m.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() * 1000 / (100 * 3600 * 24 * ocn_area)
ice_rain_avg = cm3_ice_ds.rain_ai_m.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() * 1000 / (100 * 3600 * 24 * ocn_area)
ice_sublim_avg = cm3_ice_ds.evap_ai_m.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() * 1000 / (100 * 3600 * 24 * ocn_area)
ice_melt_avg = cm3_ice_ds.fresh_ai_m.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() * 1000 / (100 * 3600 * 24 * ocn_area)

INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json
INFO:fsspec.reference:Read reference from URL /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/virtualised_outputs/seaIce.1mon.nbnd:2.nc:5.ni:1440.nj:1142.nkaer:5.nkbio:3.nkice:4.nksnow:1.json


In [74]:
def print_water_flux_error(flx1, flx2, field_name, time_slice):
    flx1 = flx1.isel(time=time_slice).mean().data
    flx2 = flx2.isel(time=time_slice).mean().data

    print(f'{field_name}: {flx1 * (3600 * 24 * 365)} mm/yr')
    print(f'{field_name} error: {(flx1 - flx2) * (3600 * 24 * 365)} mm/yr')
    print(f'{field_name} error: {(flx1 - flx2) } kg/(m^2s)')
    print(f'{field_name} relative error: {(flx1 - flx2) / flx1}')
    print()

In [83]:
time_slice = slice(0, 480)
print_water_flux_error(atm_river_avg, ocn_river_avg, "River runoff", time_slice)
print_water_flux_error(atm_evap_avg, -ocn_evap_avg, "Evaporation", time_slice)
print_water_flux_error(atm_rain_avg, ocn_rain_avg + ice_rain_avg, "Rain", time_slice)
print_water_flux_error(atm_snow_avg, ocn_snow_avg + ice_snow_avg, "Snow", time_slice)
print_water_flux_error(ice_melt_avg, ocn_melt_avg, "Melt water", time_slice)
print_water_flux_error(atm_sublim_avg, -ice_sublim_avg, "Sublimination", time_slice)

River runoff: 131.78186709645942 mm/yr
River runoff error: 0.015469663220378203 mm/yr
River runoff error: 4.905398027770866e-10 kg/(m^2s)
River runoff relative error: 0.00011738840525802375

Evaporation: 1342.5159280118569 mm/yr
Evaporation error: 0.03569303546455138 mm/yr
Evaporation error: 1.1318187298500564e-09 kg/(m^2s)
Evaporation relative error: 2.658667559900723e-05

Rain: 1123.2913053470083 mm/yr
Rain error: 0.0295172186235372 mm/yr
Rain error: 9.359848624916666e-10 kg/(m^2s)
Rain relative error: 2.627743888253342e-05

Snow: 81.28010092295777 mm/yr
Snow error: 0.0035795976595984274 mm/yr
Snow error: 1.135082971714367e-10 kg/(m^2s)
Snow relative error: 4.404027085290394e-05

Melt water: 24.593141548831017 mm/yr
Melt water error: 4.8806651439078957e-05 mm/yr
Melt water error: 1.5476487645572982e-12 kg/(m^2s)
Melt water relative error: 1.984563515082882e-06

Sublimination: 2.99000410965588 mm/yr
Sublimination error: -0.0444616839737811 mm/yr
Sublimination error: -1.409870750056478

## Energy flux error

In [ ]:
ocn_river = datastore.search(variable="friver", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).friver
ocn_evap = datastore.search(variable="evs", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).evs
ocn_snow = datastore.search(variable="prsn", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).prsn
ocn_rain = datastore.search(variable="prlq", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).prlq
ocn_total = datastore.search(variable="wfo", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).wfo